# Explainability Pipeline

Dual-layer explainability pipeline for the Hybrid Model (CNN-BiLSTM-Transformer):
1. **SHAP attribution maps** (permutation-based, per-class)
2. **NeuroKit2 clinical feature extraction** (QRS duration, PR interval, RR variability)
3. **Rule-based intersection** (checks SHAP regions against clinical criteria)
4. **Text explanation** (natural language summary per sample)

Runs on all 6 validation samples (ground truth known) and 6 test samples (unlabelled).

In [ ]:
!pip install torch numpy scipy matplotlib neurokit2 -q

In [4]:
import os
import numpy as np
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
import neurokit2 as nk
from scipy.signal import resample, butter, sosfiltfilt
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Model Definition and Loading

In [ ]:
CLASSES = ["NORM", "AFIB", "AFLT", "1dAVb", "RBBB", "LBBB", "OTHERS"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
CONFIDENCE_THRESHOLD = 0.60

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [6]:
class CNNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=7, pool_size=2):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, padding=kernel_size // 2)
        self.bn = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(pool_size)

    def forward(self, x):
        return self.pool(self.relu(self.bn(self.conv(x))))


class CNNBiLSTMTransformer(nn.Module):
    def __init__(self, num_leads=12, num_classes=7,
                 cnn_channels=None, cnn_kernel_size=7,
                 lstm_hidden=96, lstm_layers=1, lstm_dropout=0.0,
                 tf_heads=4, tf_layers=2, tf_dropout=0.3):
        super().__init__()
        if cnn_channels is None:
            cnn_channels = [48, 96, 192]

        cnn_blocks = []
        in_ch = num_leads
        for out_ch in cnn_channels:
            cnn_blocks.append(CNNBlock(in_ch, out_ch, cnn_kernel_size, pool_size=2))
            in_ch = out_ch
        self.cnn = nn.Sequential(*cnn_blocks)
        self.cnn_drop = nn.Dropout(0.2)
        self.cnn_out_dim = cnn_channels[-1]

        self.lstm = nn.LSTM(
            input_size=self.cnn_out_dim, hidden_size=lstm_hidden,
            num_layers=lstm_layers, batch_first=True, bidirectional=True,
            dropout=lstm_dropout if lstm_layers > 1 else 0.0)
        embed_dim = lstm_hidden * 2

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, 626, embed_dim) * 0.02)
        self.pos_drop = nn.Dropout(tf_dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=tf_heads, dim_feedforward=embed_dim * 4,
            dropout=tf_dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=tf_layers)
        self.norm = nn.LayerNorm(embed_dim)

        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.GELU(), nn.Dropout(tf_dropout),
            nn.Linear(embed_dim, num_classes))

    def forward(self, x):
        x = self.cnn(x)
        x = self.cnn_drop(x)
        x = x.transpose(1, 2)
        x, _ = self.lstm(x)
        B = x.shape[0]
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        seq_len = x.shape[1]
        x = self.pos_drop(x + self.pos_embed[:, :seq_len, :])
        x = self.transformer(x)
        x = self.norm(x)
        return self.head(x[:, 0])

In [ ]:
model = CNNBiLSTMTransformer(
    num_leads=12, num_classes=7,
    cnn_channels=[48, 96, 192], cnn_kernel_size=7,
    lstm_hidden=96, lstm_layers=1, lstm_dropout=0.0,
    tf_heads=4, tf_layers=2, tf_dropout=0.3
).to(DEVICE)

checkpoint_path = "../../ecg_results/results_cnn_bilstm_transformer/best_model_multilabel.pt" # please change this to the path of the best model
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True))
model.eval()

param_count = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {param_count:,} parameters")

## 2. Load and Preprocess ECG Samples

Validation and test samples are (12, 1000) at 100 Hz. The model expects (12, 5000) at 500 Hz, so we resample, filter, and normalise.

In [8]:
def bandpass_filter(signal, lowcut=0.5, highcut=50.0, fs=500, order=3):
    sos = butter(order, [lowcut, highcut], btype="band", fs=fs, output="sos")
    return sosfiltfilt(sos, signal, axis=-1)


def load_ecg_sample(npy_path):
    """Load (12,1000) 100Hz sample, resample to (12,5000) 500Hz, filter, normalise."""
    sig = np.load(npy_path).astype(np.float64)
    sig_500 = resample(sig, 5000, axis=1)
    sig_500 = np.nan_to_num(sig_500, nan=0.0)
    sig_500 = bandpass_filter(sig_500, fs=500)
    for lead in range(12):
        std = sig_500[lead].std()
        if std > 1e-6:
            sig_500[lead] = (sig_500[lead] - sig_500[lead].mean()) / std
        else:
            sig_500[lead] = 0.0
    return sig_500.astype(np.float32)

In [ ]:
VALIDATION_LABELS = {
    "validation01": "NORM",
    "validation02": "AFLT",
    "validation03": "1dAVb",
    "validation04": "AFIB",
    "validation05": "RBBB",
    "validation06": "LBBB",
}

samples = {}

for name, true_label in VALIDATION_LABELS.items():
    npy_path = f"../../../data/validation/{name}/{name}.npy"
    sig = load_ecg_sample(npy_path)
    samples[name] = {"signal": sig, "true_label": true_label, "type": "validation"}

for i in range(1, 7):
    name = f"test{i:02d}"
    npy_path = f"../../../data/test/{name}/{name}.npy"
    sig = load_ecg_sample(npy_path)
    samples[name] = {"signal": sig, "true_label": None, "type": "test"}

print(f"Loaded {len(samples)} samples")
for name, info in samples.items():
    label_str = info['true_label'] if info['true_label'] else 'unknown'
    print(f"  {name}: shape={info['signal'].shape}, true={label_str}")

## 3. Model Inference

Run the Hybrid Model on all 12 samples. Since this is a multi-label model (sigmoid output), each class has an independent probability. The predicted class is the one with the highest probability. Cases with max probability below 0.60 are flagged for clinician review.

In [ ]:
def predict(model, signal):
    with torch.no_grad():
        x = torch.FloatTensor(signal).unsqueeze(0).to(DEVICE)
        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()[0]
    return probs


for name, info in samples.items():
    probs = predict(model, info["signal"])
    pred_idx = probs.argmax()
    info["probs"] = probs
    info["pred_class"] = CLASSES[pred_idx]
    info["pred_confidence"] = float(probs[pred_idx])
    info["flagged"] = info["pred_confidence"] < CONFIDENCE_THRESHOLD

print(f"{'Sample':<16} {'True':>6} {'Predicted':>10} {'Confidence':>11} {'Flagged':>8}")
print("-" * 55)
for name, info in samples.items():
    true_str = info['true_label'] if info['true_label'] else '---'
    flag_str = 'YES' if info['flagged'] else 'no'
    match = ''
    if info['true_label']:
        match = ' OK' if info['pred_class'] == info['true_label'] else ' MISS'
    print(f"{name:<16} {true_str:>6} {info['pred_class']:>10} {info['pred_confidence']:>10.3f} {flag_str:>8}{match}")

## 4. Permutation-Based SHAP Attribution

The signal (12, 5000) is divided into temporal segments. For each segment, we estimate its Shapley value by measuring how much the class probability changes when including vs excluding that segment across random permutations. This shows which time regions the model relies on for each prediction.

In [11]:
N_SEGMENTS = 25
SAMPLES_PER_SEG = 200  # 0.4s at 500Hz
N_SHAP_SAMPLES = 50


def compute_shap_temporal(model, signal, target_class_idx, n_samples=N_SHAP_SAMPLES):
    """Permutation-based SHAP for temporal segments."""
    baseline = np.zeros_like(signal)

    def predict_fn(masks):
        batch = []
        for mask in masks:
            x = baseline.copy()
            for p in range(N_SEGMENTS):
                if mask[p] == 1:
                    s = p * SAMPLES_PER_SEG
                    e = s + SAMPLES_PER_SEG
                    x[:, s:e] = signal[:, s:e]
            batch.append(x)
        batch_t = torch.FloatTensor(np.array(batch)).to(DEVICE)
        with torch.no_grad():
            logits = model(batch_t)
            probs = torch.sigmoid(logits)
        return probs[:, target_class_idx].cpu().numpy()

    shap_values = np.zeros(N_SEGMENTS)
    rng = np.random.default_rng(42)

    for _ in range(n_samples):
        perm = rng.permutation(N_SEGMENTS)
        mask_with = np.zeros(N_SEGMENTS)
        mask_without = np.zeros(N_SEGMENTS)
        for idx in perm:
            mask_with[:] = mask_without[:]
            mask_with[idx] = 1
            val_with = predict_fn(mask_with[np.newaxis, :])[0]
            val_without = predict_fn(mask_without[np.newaxis, :])[0]
            shap_values[idx] += (val_with - val_without)
            mask_without[idx] = 1

    shap_values /= n_samples
    return shap_values


def shap_to_signal_resolution(shap_values):
    """Upsample segment-level SHAP to full signal length."""
    return np.repeat(shap_values, SAMPLES_PER_SEG)

In [ ]:
print("Computing SHAP attribution maps...")
print(f"Settings: {N_SEGMENTS} segments, {N_SHAP_SAMPLES} permutations per sample\n")

for name, info in samples.items():
    target_idx = CLASS_TO_IDX[info["pred_class"]]
    print(f"  {name} -> explaining {info['pred_class']}...", end=" ", flush=True)
    shap_vals = compute_shap_temporal(model, info["signal"], target_idx)
    info["shap_values"] = shap_vals
    info["shap_full"] = shap_to_signal_resolution(shap_vals)

    top_segs = np.argsort(np.abs(shap_vals))[::-1][:3]
    time_ranges = [(s * 0.4, (s + 1) * 0.4) for s in top_segs]
    info["shap_top_segments"] = top_segs
    info["shap_top_times"] = time_ranges
    print(f"done. Top segments: {top_segs} (times: {[f'{t[0]:.1f}-{t[1]:.1f}s' for t in time_ranges]})")

print("\nSHAP computation complete.")

## 5. NeuroKit2 Clinical Feature Extraction

For each ECG sample, NeuroKit2 processes Lead II to extract clinical parameters: heart rate, RR interval variability, QRS duration, PR interval, and P-wave detection. These measurements are the clinical ground truth that we compare against SHAP attributions.

In [13]:
def extract_clinical_features(signal_500hz):
    """Extract clinical ECG features from Lead II using NeuroKit2."""
    lead_ii = signal_500hz[1].astype(np.float64)
    fs = 500
    features = {}

    try:
        ecg_cleaned = nk.ecg_clean(lead_ii, sampling_rate=fs)
        _, rpeaks_info = nk.ecg_peaks(ecg_cleaned, sampling_rate=fs)
        r_peaks = rpeaks_info["ECG_R_Peaks"]
        features["r_peaks"] = r_peaks
        features["n_beats"] = len(r_peaks)

        if len(r_peaks) >= 2:
            rr = np.diff(r_peaks) / fs * 1000  # ms
            features["rr_mean_ms"] = float(np.mean(rr))
            features["rr_std_ms"] = float(np.std(rr))
            features["heart_rate_bpm"] = float(np.mean(60000 / rr))
            if len(rr) >= 2:
                features["rr_rmssd_ms"] = float(np.sqrt(np.mean(np.diff(rr)**2)))
            else:
                features["rr_rmssd_ms"] = None
        else:
            features["rr_mean_ms"] = None
            features["rr_std_ms"] = None
            features["heart_rate_bpm"] = None
            features["rr_rmssd_ms"] = None
    except Exception as e:
        features["r_peaks"] = []
        features["n_beats"] = 0
        features["error_peaks"] = str(e)
        return features

    try:
        _, waves = nk.ecg_delineate(ecg_cleaned, r_peaks, sampling_rate=fs, method="dwt")

        q_peaks = waves.get("ECG_Q_Peaks", [])
        s_peaks = waves.get("ECG_S_Peaks", [])
        qrs_durations = []
        for q, s in zip(q_peaks, s_peaks):
            if q is not None and s is not None and not np.isnan(q) and not np.isnan(s):
                qrs_durations.append((s - q) / fs * 1000)
        features["qrs_duration_ms"] = float(np.mean(qrs_durations)) if qrs_durations else None
        features["qrs_durations_all"] = qrs_durations

        p_onsets = waves.get("ECG_P_Onsets", [])
        p_peaks = waves.get("ECG_P_Peaks", [])
        valid_p = sum(1 for p in p_peaks if p is not None and not np.isnan(p))
        features["p_wave_count"] = valid_p
        features["p_wave_ratio"] = valid_p / max(len(r_peaks), 1)

        pr_intervals = []
        for i, p_on in enumerate(p_onsets):
            if p_on is not None and not np.isnan(p_on) and i < len(r_peaks):
                pr_ms = (r_peaks[i] - p_on) / fs * 1000
                if 50 < pr_ms < 500:
                    pr_intervals.append(pr_ms)
        features["pr_interval_ms"] = float(np.mean(pr_intervals)) if pr_intervals else None
        features["pr_intervals_all"] = pr_intervals

        t_offsets = waves.get("ECG_T_Offsets", [])
        features["t_wave_detected"] = sum(1 for t in t_offsets if t is not None and not np.isnan(t))

    except Exception as e:
        features["qrs_duration_ms"] = None
        features["pr_interval_ms"] = None
        features["p_wave_count"] = None
        features["p_wave_ratio"] = None
        features["error_delineate"] = str(e)

    return features

In [ ]:
print("Extracting clinical features with NeuroKit2...\n")

for name, info in samples.items():
    features = extract_clinical_features(info["signal"])
    info["clinical_features"] = features

print(f"{'Sample':<16} {'HR (bpm)':>10} {'QRS (ms)':>10} {'PR (ms)':>10} {'RR SD (ms)':>11} {'P-waves':>8} {'Beats':>6}")
print("-" * 75)
for name, info in samples.items():
    f = info["clinical_features"]
    hr = f"{f['heart_rate_bpm']:.0f}" if f.get('heart_rate_bpm') else "---"
    qrs = f"{f['qrs_duration_ms']:.0f}" if f.get('qrs_duration_ms') else "---"
    pr = f"{f['pr_interval_ms']:.0f}" if f.get('pr_interval_ms') else "---"
    rr_sd = f"{f['rr_std_ms']:.1f}" if f.get('rr_std_ms') else "---"
    pw = f"{f.get('p_wave_count', '---')}"
    beats = f"{f.get('n_beats', 0)}"
    print(f"{name:<16} {hr:>10} {qrs:>10} {pr:>10} {rr_sd:>11} {pw:>8} {beats:>6}")

## 6. Rule-Based Intersection Module

For each predicted condition, we check whether SHAP-highlighted regions align with the clinically expected features extracted by NeuroKit2. This cross-validation step builds clinical trust by confirming (or flagging) that the model is attending to the right parts of the signal.

In [15]:
def check_clinical_rules(pred_class, features, shap_values, signal):
    """Compare SHAP attributions against clinical diagnostic criteria."""
    checks = {}
    f = features

    if pred_class == "AFIB":
        rr_irreg = f.get("rr_rmssd_ms")
        checks["RR irregular (RMSSD > 50ms)"] = (
            rr_irreg is not None and rr_irreg > 50,
            f"RMSSD = {rr_irreg:.1f} ms" if rr_irreg else "not measurable"
        )
        p_ratio = f.get("p_wave_ratio")
        checks["P-waves absent/irregular (ratio < 0.5)"] = (
            p_ratio is not None and p_ratio < 0.5,
            f"P-wave ratio = {p_ratio:.2f}" if p_ratio is not None else "not measurable"
        )

    elif pred_class == "AFLT":
        hr = f.get("heart_rate_bpm")
        checks["Atrial rate consistent with flutter"] = (
            hr is not None and hr > 100,
            f"HR = {hr:.0f} bpm" if hr else "not measurable"
        )
        p_ratio = f.get("p_wave_ratio")
        checks["P-wave morphology abnormal"] = (
            p_ratio is not None and p_ratio < 0.7,
            f"P-wave ratio = {p_ratio:.2f}" if p_ratio is not None else "not measurable"
        )

    elif pred_class == "1dAVb":
        pr = f.get("pr_interval_ms")
        checks["PR interval > 200ms"] = (
            pr is not None and pr > 200,
            f"PR = {pr:.0f} ms" if pr else "not measurable"
        )
        # Check if SHAP highlights the PR region
        if f.get("r_peaks") is not None and len(f["r_peaks"]) > 0:
            pr_region_shap = []
            for rp in f["r_peaks"]:
                start = max(0, rp - 150)  # ~300ms before R-peak
                seg_start = start // SAMPLES_PER_SEG
                seg_end = min(rp // SAMPLES_PER_SEG, N_SEGMENTS - 1)
                pr_region_shap.extend(np.abs(shap_values[seg_start:seg_end + 1]))
            if pr_region_shap:
                mean_pr_shap = np.mean(pr_region_shap)
                mean_all_shap = np.mean(np.abs(shap_values))
                checks["SHAP highlights PR region"] = (
                    mean_pr_shap > mean_all_shap,
                    f"PR region SHAP = {mean_pr_shap:.4f} vs avg = {mean_all_shap:.4f}"
                )

    elif pred_class == "RBBB":
        qrs = f.get("qrs_duration_ms")
        checks["QRS > 120ms"] = (
            qrs is not None and qrs > 120,
            f"QRS = {qrs:.0f} ms" if qrs else "not measurable"
        )
        # Check V1-V2 lead energy for rsR' pattern
        v1_energy = np.std(signal[6])  # V1
        v2_energy = np.std(signal[7])  # V2
        avg_energy = np.mean([np.std(signal[i]) for i in range(12)])
        checks["V1-V2 morphology prominence"] = (
            (v1_energy + v2_energy) / 2 > avg_energy * 0.8,
            f"V1={v1_energy:.2f}, V2={v2_energy:.2f}, avg={avg_energy:.2f}"
        )

    elif pred_class == "LBBB":
        qrs = f.get("qrs_duration_ms")
        checks["QRS > 120ms"] = (
            qrs is not None and qrs > 120,
            f"QRS = {qrs:.0f} ms" if qrs else "not measurable"
        )
        v5_energy = np.std(signal[10])  # V5
        v6_energy = np.std(signal[11])  # V6
        avg_energy = np.mean([np.std(signal[i]) for i in range(12)])
        checks["V5-V6 morphology prominence"] = (
            (v5_energy + v6_energy) / 2 > avg_energy * 0.8,
            f"V5={v5_energy:.2f}, V6={v6_energy:.2f}, avg={avg_energy:.2f}"
        )

    elif pred_class == "NORM":
        rr_sd = f.get("rr_std_ms")
        checks["Regular rhythm (RR SD < 100ms)"] = (
            rr_sd is not None and rr_sd < 100,
            f"RR SD = {rr_sd:.1f} ms" if rr_sd else "not measurable"
        )
        hr = f.get("heart_rate_bpm")
        checks["Normal heart rate (60-100 bpm)"] = (
            hr is not None and 50 <= hr <= 110,
            f"HR = {hr:.0f} bpm" if hr else "not measurable"
        )
        qrs = f.get("qrs_duration_ms")
        checks["Normal QRS duration (< 120ms)"] = (
            qrs is not None and qrs < 120,
            f"QRS = {qrs:.0f} ms" if qrs else "not measurable"
        )

    elif pred_class == "OTHERS":
        checks["No specific clinical rule"] = (True, "classified as other abnormality")

    return checks

In [ ]:
for name, info in samples.items():
    checks = check_clinical_rules(
        info["pred_class"], info["clinical_features"],
        info["shap_values"], info["signal"]
    )
    info["clinical_checks"] = checks

    passed = sum(1 for v in checks.values() if v[0])
    total = len(checks)
    info["clinical_agreement"] = f"{passed}/{total}"

print(f"{'Sample':<16} {'Predicted':>10} {'Conf':>6} {'Clinical Checks':>16}")
print("-" * 52)
for name, info in samples.items():
    print(f"{name:<16} {info['pred_class']:>10} {info['pred_confidence']:>5.2f} {info['clinical_agreement']:>16}")
    for check_name, (passed, detail) in info["clinical_checks"].items():
        status = "PASS" if passed else "FAIL"
        print(f"    [{status}] {check_name}: {detail}")

## 7. Text Explanation Generator

Each prediction is accompanied by a natural language summary that combines the model output, SHAP findings, and NeuroKit2 measurements. A clinician can read this without needing to interpret the SHAP heatmap directly.

In [ ]:
def generate_text_explanation(name, info):
    pred = info["pred_class"]
    conf = info["pred_confidence"]
    f = info["clinical_features"]
    checks = info["clinical_checks"]
    shap_vals = info["shap_values"]

    top_seg = info["shap_top_segments"][0]
    top_time = info["shap_top_times"][0]

    lines = []
    lines.append(f"**{name}**: Predicted **{pred}** (confidence {conf:.2f}).")

    if info["flagged"]:
        lines.append("LOW CONFIDENCE: flagged for clinician review.")

    lines.append(f"SHAP highlights the {top_time[0]:.1f}--{top_time[1]:.1f}s region as most influential.")

    if f.get("heart_rate_bpm"):
        lines.append(f"Heart rate: {f['heart_rate_bpm']:.0f} bpm ({f['n_beats']} beats detected).")

    if pred == "AFIB":
        if f.get("rr_rmssd_ms"):
            lines.append(f"RR interval variability: RMSSD = {f['rr_rmssd_ms']:.1f} ms.")
        if f.get("p_wave_ratio") is not None:
            ratio = f["p_wave_ratio"]
            if ratio < 0.5:
                lines.append(f"P-waves largely absent (ratio {ratio:.2f}), consistent with AFIB.")
            else:
                lines.append(f"P-waves detected (ratio {ratio:.2f}), which may not fully support AFIB.")

    elif pred == "AFLT":
        lines.append("Look for sawtooth flutter waves in leads II, III, and aVF.")
        if f.get("heart_rate_bpm") and f["heart_rate_bpm"] > 100:
            lines.append(f"Elevated rate ({f['heart_rate_bpm']:.0f} bpm) is consistent with atrial flutter.")

    elif pred == "1dAVb":
        if f.get("pr_interval_ms"):
            pr = f["pr_interval_ms"]
            if pr > 200:
                lines.append(f"NeuroKit2 confirms PR interval of {pr:.0f} ms (threshold: 200 ms). Consistent with 1dAVb.")
            else:
                lines.append(f"NeuroKit2 measured PR interval of {pr:.0f} ms (below 200 ms threshold).")

    elif pred == "RBBB":
        if f.get("qrs_duration_ms"):
            qrs = f["qrs_duration_ms"]
            if qrs > 120:
                lines.append(f"NeuroKit2 confirms QRS duration of {qrs:.0f} ms (threshold: 120 ms). Consistent with RBBB.")
            else:
                lines.append(f"QRS duration measured at {qrs:.0f} ms (below 120 ms threshold).")
        lines.append("Check for rsR' pattern in leads V1-V2.")

    elif pred == "LBBB":
        if f.get("qrs_duration_ms"):
            qrs = f["qrs_duration_ms"]
            if qrs > 120:
                lines.append(f"NeuroKit2 confirms QRS duration of {qrs:.0f} ms (threshold: 120 ms). Consistent with LBBB.")
            else:
                lines.append(f"QRS duration measured at {qrs:.0f} ms (below 120 ms threshold).")
        lines.append("Check for broad, notched R-waves in leads V5-V6.")

    elif pred == "NORM":
        lines.append("No abnormal features detected by NeuroKit2.")

    passed = sum(1 for v in checks.values() if v[0])
    total = len(checks)
    if total > 0:
        if passed == total:
            lines.append(f"Clinical validation: all {total} checks passed.")
        else:
            lines.append(f"Clinical validation: {passed}/{total} checks passed. Review recommended.")

    return " ".join(lines)


print("=" * 80)
print("TEXT EXPLANATIONS")
print("=" * 80)
for name, info in samples.items():
    explanation = generate_text_explanation(name, info)
    info["text_explanation"] = explanation
    print(f"\n{explanation}")

## 8. SHAP Visualisation

Each plot overlays the SHAP attribution heatmap on the ECG waveform (Lead II). Warmer colours indicate higher SHAP values (regions the model relied on most for its prediction). NeuroKit2 R-peak locations are marked for reference.

In [18]:
OUTPUT_DIR = "../../report/figures/explainability"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def plot_shap_ecg(name, info, save=True):
    signal = info["signal"]
    shap_full = info["shap_full"]
    f = info["clinical_features"]
    pred = info["pred_class"]
    conf = info["pred_confidence"]
    true_label = info["true_label"]

    shap_abs = np.abs(shap_full)
    if shap_abs.max() > 0:
        shap_norm = shap_abs / shap_abs.max()
    else:
        shap_norm = shap_abs

    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True,
                             gridspec_kw={"height_ratios": [2, 2, 1]})

    t = np.arange(5000) / 500  # seconds

    # Panel 1: Lead II with SHAP overlay
    ax = axes[0]
    ax.plot(t, signal[1], color="black", linewidth=0.6, alpha=0.7)
    cmap = plt.get_cmap("YlOrRd")
    for i in range(len(t) - 1):
        val = shap_norm[i]
        if val > 0.05:
            ax.axvspan(t[i], t[i+1], color=cmap(val), alpha=val * 0.6)
    if f.get("r_peaks") is not None:
        for rp in f["r_peaks"]:
            ax.axvline(rp / 500, color="blue", alpha=0.3, linewidth=0.8, linestyle="--")
    ax.set_ylabel("Lead II")
    title = f"{name} | Predicted: {pred} ({conf:.2f})"
    if true_label:
        title += f" | True: {true_label}"
    if info["flagged"]:
        title += " | FLAGGED"
    ax.set_title(title, fontsize=11, fontweight="bold")

    # Panel 2: V1 with SHAP overlay (useful for BBB)
    ax = axes[1]
    ax.plot(t, signal[6], color="black", linewidth=0.6, alpha=0.7)
    for i in range(len(t) - 1):
        val = shap_norm[i]
        if val > 0.05:
            ax.axvspan(t[i], t[i+1], color=cmap(val), alpha=val * 0.6)
    ax.set_ylabel("Lead V1")

    # Panel 3: SHAP bar chart per segment
    ax = axes[2]
    seg_times = np.arange(N_SEGMENTS) * 0.4 + 0.2  # center of each segment
    colors = ["#d73027" if v > 0 else "#4575b4" for v in info["shap_values"]]
    ax.bar(seg_times, info["shap_values"], width=0.38, color=colors, alpha=0.8)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.set_ylabel("SHAP value")
    ax.set_xlabel("Time (seconds)")

    plt.tight_layout()
    if save:
        path = os.path.join(OUTPUT_DIR, f"shap_{name}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        print(f"Saved: {path}")
    plt.show()
    plt.close()

In [ ]:
print("Generating SHAP overlay plots...\n")
for name, info in samples.items():
    plot_shap_ecg(name, info)

## 9. Multi-Lead SHAP View

For selected samples, show SHAP attribution across all 12 leads to see which leads the model focuses on.

In [ ]:
def plot_12lead_shap(name, info, save=True):
    signal = info["signal"]
    shap_full = info["shap_full"]
    pred = info["pred_class"]
    conf = info["pred_confidence"]
    true_label = info["true_label"]

    shap_abs = np.abs(shap_full)
    shap_norm = shap_abs / shap_abs.max() if shap_abs.max() > 0 else shap_abs

    fig, axes = plt.subplots(12, 1, figsize=(14, 16), sharex=True)
    t = np.arange(5000) / 500
    cmap = plt.get_cmap("YlOrRd")

    for lead_idx in range(12):
        ax = axes[lead_idx]
        ax.plot(t, signal[lead_idx], color="black", linewidth=0.4, alpha=0.7)
        for i in range(0, len(t) - 1, 2):
            val = shap_norm[i]
            if val > 0.05:
                ax.axvspan(t[i], t[min(i+2, len(t)-1)], color=cmap(val), alpha=val * 0.5)
        ax.set_ylabel(LEAD_NAMES[lead_idx], rotation=0, labelpad=20, fontsize=9)
        ax.tick_params(labelsize=6)

    title = f"{name} | All 12 Leads | Predicted: {pred} ({conf:.2f})"
    if true_label:
        title += f" | True: {true_label}"
    fig.suptitle(title, fontsize=12, fontweight="bold")
    axes[-1].set_xlabel("Time (seconds)")
    plt.tight_layout()

    if save:
        path = os.path.join(OUTPUT_DIR, f"shap_12lead_{name}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        print(f"Saved: {path}")
    plt.show()
    plt.close()


selected = ["validation05", "validation06", "validation03"]
for name in selected:
    if name in samples:
        plot_12lead_shap(name, samples[name])

## 10. NeuroKit2 Waveform Annotation

Overlay NeuroKit2 detected features (R-peaks, P-waves, QRS boundaries) on the ECG to show the clinical feature extraction layer.

In [ ]:
def plot_neurokit_annotation(name, info, save=True):
    signal = info["signal"]
    f = info["clinical_features"]
    lead_ii = signal[1]
    t = np.arange(5000) / 500

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(t, lead_ii, color="black", linewidth=0.7, label="Lead II")

    if f.get("r_peaks") is not None:
        for rp in f["r_peaks"]:
            ax.plot(rp / 500, lead_ii[min(rp, 4999)], "rv", markersize=8, alpha=0.8)

    legend_items = [mpatches.Patch(color="red", label="R-peaks")]

    pred = info["pred_class"]
    conf = info["pred_confidence"]
    true_label = info["true_label"]

    feature_text = []
    if f.get("heart_rate_bpm"):
        feature_text.append(f"HR: {f['heart_rate_bpm']:.0f} bpm")
    if f.get("qrs_duration_ms"):
        feature_text.append(f"QRS: {f['qrs_duration_ms']:.0f} ms")
    if f.get("pr_interval_ms"):
        feature_text.append(f"PR: {f['pr_interval_ms']:.0f} ms")
    if f.get("rr_std_ms"):
        feature_text.append(f"RR SD: {f['rr_std_ms']:.1f} ms")

    if feature_text:
        ax.text(0.02, 0.95, " | ".join(feature_text),
                transform=ax.transAxes, fontsize=9, verticalalignment="top",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8))

    title = f"{name} | NeuroKit2 Features | Predicted: {pred} ({conf:.2f})"
    if true_label:
        title += f" | True: {true_label}"
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Amplitude")
    ax.legend(handles=legend_items, loc="upper right", fontsize=8)

    plt.tight_layout()
    if save:
        path = os.path.join(OUTPUT_DIR, f"neurokit_{name}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()


for name, info in list(samples.items())[:4]:
    plot_neurokit_annotation(name, info)

## 11. Summary Results

Final summary table with predictions, confidence, clinical checks, and text explanations for all 12 samples.

In [ ]:
print("\n" + "=" * 100)
print("VALIDATION SET RESULTS (ground truth available)")
print("=" * 100)
print(f"{'Sample':<16} {'True':>6} {'Predicted':>10} {'Conf':>6} {'Match':>6} {'Flag':>5} {'Checks':>8}")
print("-" * 60)
correct = 0
for name, info in samples.items():
    if info["type"] != "validation":
        continue
    match = "YES" if info["pred_class"] == info["true_label"] else "NO"
    if match == "YES":
        correct += 1
    flag = "YES" if info["flagged"] else "no"
    print(f"{name:<16} {info['true_label']:>6} {info['pred_class']:>10} "
          f"{info['pred_confidence']:>5.2f} {match:>6} {flag:>5} {info['clinical_agreement']:>8}")
print(f"\nValidation accuracy: {correct}/6")

print("\n" + "=" * 100)
print("TEST SET PREDICTIONS (unlabelled)")
print("=" * 100)
print(f"{'Sample':<16} {'Predicted':>10} {'Conf':>6} {'Flag':>5} {'Checks':>8}")
print("-" * 50)
for name, info in samples.items():
    if info["type"] != "test":
        continue
    flag = "YES" if info["flagged"] else "no"
    print(f"{name:<16} {info['pred_class']:>10} {info['pred_confidence']:>5.2f} {flag:>5} {info['clinical_agreement']:>8}")

In [ ]:
print("\n" + "=" * 100)
print("TEXT EXPLANATIONS (for report Section 4.4)")
print("=" * 100)
for name, info in samples.items():
    print(f"\n--- {name} ---")
    print(info["text_explanation"])

In [ ]:
print(f"\nAll figures saved to: {os.path.abspath(OUTPUT_DIR)}")
print(f"Total figures: {len([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.png')])}")

## 12. Unified Clinical Visualization

One figure per sample that combines all pipeline outputs in a single readable view:
- **Only the diagnostically relevant leads** per predicted class (not all 12)
- **Top-2 SHAP segments** highlighted as background bands (dark red = top-1, orange = top-2)
- **Class-specific NeuroKit2 markers** drawn on the ECG:
  - `1dAVb` → PR interval brackets (P-onset to R-peak) on Lead II
  - `RBBB`  → QRS onset/offset lines on V1, V2
  - `LBBB`  → QRS onset/offset lines on V5, V6
  - `AFIB`  → R-peak irregularity visible + extra RR interval series panel
  - `AFLT`  → R-peak markers + HR annotation on II, III, aVF
  - `NORM`  → R-peak markers confirming regular rhythm

Existing cells (Sections 8–11) are unchanged. Figures saved to `unified/` subdirectory.

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import AutoMinorLocator

UNIFIED_DIR = os.path.join(OUTPUT_DIR, "unified")
os.makedirs(UNIFIED_DIR, exist_ok=True)

CLASS_LEADS = {
    "NORM":   {"leads": [1],         "names": ["Lead II"]},
    "1dAVb":  {"leads": [1],         "names": ["Lead II"]},
    "AFIB":   {"leads": [1],         "names": ["Lead II"]},
    "AFLT":   {"leads": [1, 2, 5],   "names": ["Lead II", "Lead III", "Lead aVF"]},
    "RBBB":   {"leads": [1, 6, 7],   "names": ["Lead II", "Lead V1", "Lead V2"]},
    "LBBB":   {"leads": [0, 10, 11], "names": ["Lead I",  "Lead V5",  "Lead V6"]},
    "OTHERS": {"leads": [1],         "names": ["Lead II"]},
}


def get_visual_markers(signal, pred_class, r_peaks, fs=500):
    """Extract class-specific visual markers for the unified plot."""
    m = {"r_peaks": r_peaks}

    if pred_class == "1dAVb":
        try:
            cleaned = nk.ecg_clean(signal[1].astype(np.float64), sampling_rate=fs)
            _, waves = nk.ecg_delineate(cleaned, r_peaks, sampling_rate=fs, method="dwt")
            raw = waves.get("ECG_P_Onsets", [])
            m["p_onsets"] = [int(p) for p in raw if p is not None and not np.isnan(p)]
        except Exception:
            m["p_onsets"] = []

    elif pred_class == "AFIB":
        if len(r_peaks) >= 2:
            rr = np.diff(r_peaks) / fs * 1000
            m["rr_ms"]    = rr
            m["rr_times"] = r_peaks[1:] / fs
        else:
            m["rr_ms"]    = np.array([])
            m["rr_times"] = np.array([])

    # RBBB and LBBB: QRS windows are now computed from R-peaks + measured QRS duration
    # inside plot_unified_clinical — no per-lead delineation needed here.

    return m


def plot_unified_clinical(name, info, save=True):
    pred       = info["pred_class"]
    conf       = info["pred_confidence"]
    true_label = info["true_label"]
    signal     = info["signal"]
    f          = info["clinical_features"]
    checks     = info["clinical_checks"]
    r_peaks    = np.array(f.get("r_peaks", []), dtype=int)

    lead_cfg     = CLASS_LEADS.get(pred, CLASS_LEADS["NORM"])
    lead_indices = lead_cfg["leads"]
    lead_names   = lead_cfg["names"]

    # Increased alpha for better visibility
    top2        = info["shap_top_segments"][:2]
    shap_colors = ["#d73027", "#f46d43"]
    shap_alphas = [0.50, 0.35]

    markers  = get_visual_markers(signal, pred, r_peaks)
    has_rr   = (pred == "AFIB") and len(markers.get("rr_ms", [])) > 0
    n_leads  = len(lead_indices)
    n_panels = n_leads + (1 if has_rr else 0)
    h_ratios = [2] * n_leads + ([1] if has_rr else [])
    t        = np.arange(5000) / 500

    fig, axes = plt.subplots(
        n_panels, 1,
        figsize=(14, 3 * n_panels + 1),
        sharex=True,
        gridspec_kw={"height_ratios": h_ratios}
    )
    if n_panels == 1:
        axes = [axes]

    # Pre-compute QRS window offsets for RBBB/LBBB (consistent across all leads)
    if pred in ("RBBB", "LBBB"):
        qrs_ms  = f.get("qrs_duration_ms") or 100
        fs_val  = 500
        # ~40% of QRS is before R-peak, ~60% after (typical QRS morphology)
        q_samp  = int(qrs_ms * 0.40 / 1000 * fs_val)
        s_samp  = int(qrs_ms * 0.60 / 1000 * fs_val)

    # ── ECG lead panels ─────────────────────────────────────────────
    for pi, (lidx, lname) in enumerate(zip(lead_indices, lead_names)):
        ax  = axes[pi]
        ecg = signal[lidx]

        ax.plot(t, ecg, color=(0, 0, 0.7), linewidth=0.8)
        # Top-2 SHAP highlights — stronger alpha
        for rank, seg in enumerate(top2):
            ax.axvspan(seg * 0.4, (seg + 1) * 0.4,
                       color=shap_colors[rank], alpha=shap_alphas[rank],
                       label=f"SHAP top-{rank + 1}" if pi == 0 else "")

        # R-peak markers
        for rp in r_peaks:
            if rp < len(ecg):
                ax.plot(rp / 500, ecg[rp], "v",
                        color="steelblue", markersize=6, alpha=0.85,
                        label="R-peak" if (pi == 0 and rp == r_peaks[0]) else "")

        # ── class-specific overlays ──────────────────────────────────

        if pred == "1dAVb" and pi == 0:
            p_onsets = markers.get("p_onsets", [])
            y_br = float(np.min(ecg)) - 0.3
            for i, rp in enumerate(r_peaks):
                if i < len(p_onsets) and p_onsets[i] < len(ecg):
                    ax.annotate(
                        "", xy=(rp / 500, y_br), xytext=(p_onsets[i] / 500, y_br),
                        arrowprops=dict(arrowstyle="<->", color="green", lw=1.8)
                    )
            pr = f.get("pr_interval_ms")
            if pr:
                ax.text(0.02, 0.06,
                        f"PR = {pr:.0f} ms  (threshold: 200 ms → 1dAVb confirmed)",
                        transform=ax.transAxes, fontsize=9, color="green",
                        bbox=dict(boxstyle="round,pad=0.2", facecolor="honeydew", alpha=0.85))

        elif pred in ("RBBB", "LBBB") and pi > 0:
            # Shaded QRS region around each R-peak — same positions across all leads
            for rp in r_peaks:
                t_q = max(0.0, (rp - q_samp) / 500)
                t_s = min(10.0, (rp + s_samp) / 500)
                ax.axvspan(t_q, t_s, color="purple", alpha=0.18, zorder=0)

            if pi == 1:
                qrs = f.get("qrs_duration_ms")
                if qrs:
                    if qrs > 120:
                        label = f"QRS = {qrs:.0f} ms  (> 120 ms threshold → {pred} confirmed)"
                    else:
                        label = f"QRS = {qrs:.0f} ms  (< 120 ms — resampling artefact; V1/V2 morphology supports {pred})"
                    ax.text(0.02, 0.06, label,
                            transform=ax.transAxes, fontsize=9, color="purple",
                            bbox=dict(boxstyle="round,pad=0.2", facecolor="lavender", alpha=0.85))

        elif pred == "AFLT" and pi == 0:
            hr = f.get("heart_rate_bpm")
            if hr:
                ax.text(0.02, 0.06,
                        f"HR = {hr:.0f} bpm  (> 100 bpm consistent with atrial flutter — "
                        f"look for sawtooth pattern in II, III, aVF)",
                        transform=ax.transAxes, fontsize=9, color="#7b2d8b",
                        bbox=dict(boxstyle="round,pad=0.2", facecolor="thistle", alpha=0.85))

        elif pred == "NORM" and pi == 0:
            rr_sd = f.get("rr_std_ms")
            if rr_sd:
                ax.text(0.02, 0.06,
                        f"RR SD = {rr_sd:.1f} ms  (< 100 ms → regular rhythm confirmed)",
                        transform=ax.transAxes, fontsize=9, color="#1a6e1a",
                        bbox=dict(boxstyle="round,pad=0.2", facecolor="honeydew", alpha=0.85))

        ax.set_ylabel(lname, fontsize=10)
        ax.set_xlim(0, 10)
        # ECG paper grid
        ax.set_xticks(np.arange(0, 10.01, 0.2))
        ax.set_yticks(np.arange(np.floor(ecg.min()), np.ceil(ecg.max()) + 0.5, 0.5))
        ax.xaxis.set_minor_locator(AutoMinorLocator(5))
        ax.yaxis.set_minor_locator(AutoMinorLocator(5))
        ax.set_xticklabels([f"{int(x)}s" if x % 1 == 0 else "" for x in np.arange(0, 10.01, 0.2)])
        ax.grid(which='major', linestyle='-', linewidth=0.5, color=(1, 0, 0), alpha=0.5)
        ax.grid(which='minor', linestyle='-', linewidth=0.25, color=(1, 0.7, 0.7), alpha=0.5)
        ax.set_axisbelow(True)


    # ── RR interval panel (AFIB only) ────────────────────────────────
    if has_rr:
        ax_rr = axes[n_leads]
        rr_ms = markers["rr_ms"]
        rr_t  = markers["rr_times"]
        ax_rr.plot(rr_t, rr_ms, "o-", color="#e31a1c", markersize=4, linewidth=1.2)
        ax_rr.axhline(np.mean(rr_ms), color="gray", linewidth=0.8, linestyle="--",
                      label=f"Mean RR = {np.mean(rr_ms):.0f} ms")
        rmssd = f.get("rr_rmssd_ms")
        if rmssd:
            ax_rr.text(0.02, 0.78,
                       f"RMSSD = {rmssd:.1f} ms  (> 50 ms → irregular rhythm confirms AFIB)",
                       transform=ax_rr.transAxes, fontsize=9, color="#e31a1c",
                       bbox=dict(boxstyle="round,pad=0.2", facecolor="mistyrose", alpha=0.85))
        ax_rr.legend(fontsize=8, loc="upper right")
        ax_rr.set_ylabel("RR (ms)", fontsize=10)

    axes[-1].set_xlabel("Time (seconds)", fontsize=10)

    # ── Title ────────────────────────────────────────────────────────
    passed = sum(1 for v in checks.values() if v[0])
    total  = len(checks)
    title  = f"{name}  |  Predicted: {pred} ({conf:.2f})"
    if true_label:
        title += f"  |  True: {true_label}"
    title += f"  |  Clinical checks: {passed}/{total}"
    if info["flagged"]:
        title += "  |  FLAGGED"
    axes[0].set_title(title, fontsize=11, fontweight="bold")

    # ── Legend ───────────────────────────────────────────────────────
    legend_els = [
        Patch(facecolor="#d73027", alpha=0.6, label="SHAP top-1 region"),
        Patch(facecolor="#f46d43", alpha=0.5, label="SHAP top-2 region"),
        Line2D([0], [0], marker="v", color="w",
               markerfacecolor="steelblue", markersize=8, label="R-peak"),
    ]
    if pred == "1dAVb":
        legend_els.append(
            Line2D([0], [0], color="green", lw=1.8, label="PR interval bracket"))
    elif pred in ("RBBB", "LBBB"):
        legend_els.append(
            Patch(facecolor="purple", alpha=0.3, label="QRS region"))
    axes[0].legend(handles=legend_els, loc="upper right", fontsize=8, framealpha=0.85)

    plt.tight_layout()
    if save:
        path = os.path.join(UNIFIED_DIR, f"unified_{name}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        print(f"Saved: {path}")
    plt.show()
    plt.close()


print("Generating unified clinical visualizations...\n")
for name, info in samples.items():
    plot_unified_clinical(name, info)
print(f"\nAll unified figures saved to: {UNIFIED_DIR}")

## 13. Raw Signal Visualization (Original ECG + SHAP + NeuroKit2)

Same layout as Section 12 but using the **original raw .npy signal (100 Hz, no resampling, no normalisation)** for display.
This produces a waveform that matches what a clinician would see in the original ECG printout.

SHAP segments and NeuroKit2 markers are mapped back to the original time axis using time-in-seconds
(segment boundaries are purely temporal — they apply to any sampling rate).

Figures saved to `unified/raw_*.png`.

In [ ]:
try:
    import ecg_plot
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "ecg-plot", "-q"], check=True)
    import ecg_plot


def load_raw_signal(name, sample_type):
    base = "../../../data/validation" if sample_type == "validation" else "../../../data/test"
    return np.load(f"{base}/{name}/{name}.npy").astype(np.float32)


def plot_12lead_shap(name, info, save=True):
    pred        = info["pred_class"]
    conf        = info["pred_confidence"]
    true_label  = info["true_label"]
    f           = info["clinical_features"]
    checks      = info["clinical_checks"]
    r_peaks_500 = np.array(f.get("r_peaks", []), dtype=int)
    r_peaks_sec = r_peaks_500 / 500.0

    raw_sig = load_raw_signal(name, info["type"])
    ecg     = raw_sig.T   # (1000, 12)

    top2        = info["shap_top_segments"][:2]
    shap_colors = ["#d73027", "#f46d43"]

    # Lead concatenation (same as plot_ecg_samples.ipynb)
    n = ecg.shape[0]
    q1, q2, q3 = n//4, n//2, 3*n//4
    lead_I_new   = np.concatenate([ecg[0:q1,0],  ecg[q1:q2,3],  ecg[q2:q3,6],  ecg[q3:n,9]])
    lead_II_new  = np.concatenate([ecg[0:q1,1],  ecg[q1:q2,4],  ecg[q2:q3,7],  ecg[q3:n,10]])
    lead_III_new = np.concatenate([ecg[0:q1,2],  ecg[q1:q2,5],  ecg[q2:q3,8],  ecg[q3:n,11]])
    lead_IV_new  = ecg[:, 1]
    ecg_new = np.stack([lead_I_new, lead_II_new, lead_III_new, lead_IV_new], axis=1).T

    passed = sum(1 for v in checks.values() if v[0])
    total  = len(checks)
    title  = f"{name}  |  Predicted: {pred} ({conf:.2f})"
    if true_label: title += f"  |  True: {true_label}"
    title += f"  |  Clinical checks: {passed}/{total}"

    ecg_plot.plot(ecg_new, sample_rate=100, title=title,
                  columns=1, lead_index=['I', 'II', 'III', 'II Ref'])
    ax = plt.gca()

    # Dynamic row fractions from actual y-limits (robust to any signal amplitude)
    y0, y1 = ax.get_ylim()
    h = (y1 - y0) / 4.0
    def to_frac(y): return (y - y0) / (y1 - y0)
    ROW_YFRACS = {
        0: (to_frac(y1 - h),    1.0),
        1: (to_frac(y1 - 2*h),  to_frac(y1 - h)),
        2: (to_frac(y1 - 3*h),  to_frac(y1 - 2*h)),
        3: (0.0,                 to_frac(y1 - 3*h)),
    }

    # Lead column labels
    ax.text(2.55, -0.6,  'aVR', fontsize=8, color='black')
    ax.text(2.55, -4.1,  'aVL', fontsize=8, color='black')
    ax.text(2.55, -7.6,  'aVF', fontsize=8, color='black')
    ax.text(5.05, -0.6,  'V1',  fontsize=8, color='black')
    ax.text(5.05, -4.1,  'V2',  fontsize=8, color='black')
    ax.text(5.05, -7.6,  'V3',  fontsize=8, color='black')
    ax.text(7.55, -0.6,  'V4',  fontsize=8, color='black')
    ax.text(7.55, -4.1,  'V5',  fontsize=8, color='black')
    ax.text(7.55, -7.6,  'V6',  fontsize=8, color='black')
    ax.text(0.1, ax.get_ylim()[0] + 0.2, '100 Hz   25.0 mm/s   10.0mm/mV', fontsize=9)

    # Column -> lead per row in the 12-lead layout
    COL_ROW_LEAD = {
        0: {0: "I",   1: "II",  2: "III"},
        1: {0: "aVR", 1: "aVL", 2: "aVF"},
        2: {0: "V1",  1: "V2",  2: "V3"},
        3: {0: "V4",  1: "V5",  2: "V6"},
    }
    DIAG_LEADS_MAP = {
        "NORM":  {"II"},
        "1dAVb": {"II"},
        "AFIB":  {"II"},
        "AFLT":  {"II", "III", "aVF"},
        "RBBB":  {"II", "V1", "V2"},
        "LBBB":  {"I",  "V5", "V6"},
    }
    diag_leads = DIAG_LEADS_MAP.get(pred, {"II"})

    # SHAP: light on full column, dark only where diagnostic lead is visible at that column
    for rank, seg in enumerate(top2):
        t0  = seg * 0.4
        t1  = (seg + 1) * 0.4
        col = min(int(t0 / 2.5), 3)

        # Layer 1: light across full height (context)
        ax.axvspan(t0, t1, color=shap_colors[rank], alpha=0.12, zorder=5)

        # Layer 2: dark only on rows where a diagnostic lead is shown at this column
        for row, lead in COL_ROW_LEAD.get(col, {}).items():
            if lead in diag_leads:
                ymin_f, ymax_f = ROW_YFRACS[row]
                ax.axvspan(t0, t1, ymin=ymin_f, ymax=ymax_f,
                           color=shap_colors[rank], alpha=0.45, zorder=6)

        # Layer 2b: rhythm strip -- dark if II is diagnostic for this class
        if "II" in diag_leads:
            ymin_r, ymax_r = ROW_YFRACS[3]
            ax.axvspan(t0, t1, ymin=ymin_r, ymax=ymax_r,
                       color=shap_colors[rank], alpha=0.35, zorder=6)

    # R-peaks on II Ref (rhythm strip only)
    y_rhythm = ax.get_ylim()[0] + abs(ax.get_ylim()[0]) * 0.12
    for rp_s in r_peaks_sec:
        ax.plot(rp_s, y_rhythm, "v", color="steelblue",
                markersize=5, alpha=0.85, zorder=7)

    # Clinical annotation bar
    parts = []
    if f.get("heart_rate_bpm"):  parts.append(f"HR: {f['heart_rate_bpm']:.0f} bpm")
    if f.get("qrs_duration_ms"): parts.append(f"QRS: {f['qrs_duration_ms']:.0f} ms")
    if f.get("pr_interval_ms"):  parts.append(f"PR: {f['pr_interval_ms']:.0f} ms")
    if f.get("rr_std_ms"):       parts.append(f"RR SD: {f['rr_std_ms']:.1f} ms")
    if f.get("rr_rmssd_ms"):     parts.append(f"RMSSD: {f['rr_rmssd_ms']:.1f} ms")
    ax.text(0.5, -0.07, "  |  ".join(parts), transform=ax.transAxes,
            fontsize=8, ha="center",
            bbox=dict(facecolor="lightyellow", alpha=0.85, pad=3, edgecolor="none"))

    # Legend
    legend_els = [
        Patch(facecolor="#d73027", alpha=0.4, label="SHAP top-1 (context)"),
        Patch(facecolor="#d73027", alpha=0.9, label=f"SHAP top-1 (diagnostic: {pred})"),
        Patch(facecolor="#f46d43", alpha=0.3, label="SHAP top-2 (context)"),
        Line2D([0],[0], marker="v", color="w", markerfacecolor="steelblue",
               markersize=7, label="R-peak (II Ref)"),
    ]
    ax.legend(handles=legend_els, loc="upper right", fontsize=7, framealpha=0.9)

    if save:
        path = os.path.join(UNIFIED_DIR, f"raw12lead_{name}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        print(f"Saved: {path}")
    ecg_plot.show()
    plt.close()


print("Generating standard 12-lead + SHAP visualizations...\n")
for name, info in samples.items():
    plot_12lead_shap(name, info)
print(f"\nAll 12-lead figures saved to: {UNIFIED_DIR}")


## Cell 14: Hybrid Per-Lead SHAP — Enhanced Visualization

Computes **leave-one-out per lead** attribution for the top-3 SHAP time segments
(~48 forward passes per sample vs 2,500 for full temporal SHAP).

Answers: *which leads* did the model focus on in those important moments?

Two new figures per sample saved to `unified/hybrid/`:
- **hybrid12lead_**: 12-lead layout, light SHAP bands + colored Rectangle border per hybrid lead. Double-mark when time aligns with that lead's column.
- **hybrid_detail_**: diagnostic + hybrid leads color-coded (red = medical expected **and** model used; orange = model-only; grey = medical-only)

Cells 12 and 13 are unchanged.

In [ ]:
# ─── Cell 14: Hybrid Per-Lead SHAP + Enhanced Visualization ──────────────────
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

# ── Layout constants ──────────────────────────────────────────────────────────
# 12-lead ecg_plot format: 4 cols × 3 rows + rhythm strip (Lead II)
LEAD_CELL_MAP = {
    "I":   (0, 0), "II":  (0, 1), "III": (0, 2),
    "aVR": (1, 0), "aVL": (1, 1), "aVF": (1, 2),
    "V1":  (2, 0), "V2":  (2, 1), "V3":  (2, 2),
    "V4":  (3, 0), "V5":  (3, 1), "V6":  (3, 2),
}
COL_X = {0: (0.0, 2.5), 1: (2.5, 5.0), 2: (5.0, 7.5), 3: (7.5, 10.0)}

DIAG_LEADS_H = {
    "NORM":  {"II"},
    "1dAVb": {"II"},
    "AFIB":  {"II"},
    "AFLT":  {"II", "III", "aVF"},
    "RBBB":  {"II", "V1", "V2"},
    "LBBB":  {"I",  "V5", "V6"},
}

_SC3 = ["#d73027", "#f46d43", "#fdae61"]   # SHAP top-1/2/3 colours

HYBRID_DIR = os.path.join(UNIFIED_DIR, "hybrid")
os.makedirs(HYBRID_DIR, exist_ok=True)


# ── Hybrid computation ────────────────────────────────────────────────────────

def compute_hybrid_lead_importance(sig500, target_class_idx, top_segs):
    # Leave-one-out per lead for each top segment. ~48 forward passes total.
    # Uses raw logits (not sigmoid) to avoid saturation when confidence ~1.0:
    # sigmoid(8)=0.9997 and sigmoid(6)=0.9975 both round to 1.0 in float32,
    # making all differences zero. Logit differences remain meaningful.
    scores = np.zeros(12)
    with torch.no_grad():
        base_logit = model(
            torch.FloatTensor(sig500[np.newaxis]).to(DEVICE)
        )[0, target_class_idx].item()
    for seg in top_segs:
        s = seg * SAMPLES_PER_SEG
        e = min(s + SAMPLES_PER_SEG, sig500.shape[1])
        for li in range(12):
            x = sig500.copy()
            x[li, s:e] = 0.0
            with torch.no_grad():
                logit = model(
                    torch.FloatTensor(x[np.newaxis]).to(DEVICE)
                )[0, target_class_idx].item()
            scores[li] += (base_logit - logit)
    return scores  # positive = lead helped; works even when sigmoid saturates


# Run hybrid computation for all samples
print("Computing hybrid lead importance (leave-one-out per lead x top-3 segments)...")
hybrid_results = {}
for name, info in samples.items():
    pred   = info["pred_class"]
    cidx   = CLASS_TO_IDX.get(pred, 0)
    top3   = info["shap_top_segments"][:3]
    scores = compute_hybrid_lead_importance(info["signal"], cidx, top3)
    order  = np.argsort(scores)[::-1]
    top3n  = [LEAD_NAMES[i] for i in order[:3]]
    hybrid_results[name] = {"scores": scores, "top3_names": top3n, "top3_segs": top3}
    print(f"  {name:14s} ({pred:5s}): top hybrid leads = {top3n}  "
          f"scores = {scores[order[:3]].round(4)}")
print()


# ── Geometry helper ───────────────────────────────────────────────────────────

def _row_geom(ax):
    # Return (bottoms, tops, yfracs, h) for the 4 rows in a 12-lead ecg_plot axis.
    y0, y1 = ax.get_ylim()
    h = (y1 - y0) / 4.0
    tops    = [y1 - i * h       for i in range(4)]
    bottoms = [y1 - (i + 1) * h for i in range(4)]
    def frac(y): return (y - y0) / (y1 - y0)
    yfracs  = [(frac(bottoms[r]), frac(tops[r])) for r in range(4)]
    return bottoms, tops, yfracs, h


def _role(lead, med_leads, hyb_set):
    # Return (edge_color, linestyle, linewidth, fill_alpha) for a lead cell border.
    is_m = lead in med_leads
    is_h = lead in hyb_set
    if   is_m and is_h: return "#d73027", "-",  2.0, 0.38
    elif is_h:          return "#f46d43", "-",  1.5, 0.32
    else:               return "#888888", "--", 1.0, 0.10   # medical-only


# ── Plot A: Enhanced 12-lead ──────────────────────────────────────────────────

def plot_hybrid_12lead(name, info, h_res, save=True):
    pred        = info["pred_class"]
    conf        = info["pred_confidence"]
    true_label  = info["true_label"]
    f           = info["clinical_features"]
    checks      = info["clinical_checks"]
    r_peaks_sec = np.array(f.get("r_peaks", []), dtype=int) / 500.0

    scores      = h_res["scores"]
    top3_segs   = h_res["top3_segs"]
    top3_hyb    = set(h_res["top3_names"])
    med_leads   = DIAG_LEADS_H.get(pred, {"II"})

    # Build 12-lead display ECG (100 Hz raw, same layout as Cell 13)
    raw = load_raw_signal(name, info["type"])   # (12, 1000)
    ecg = raw.T                                  # (1000, 12)
    n = ecg.shape[0]
    q1, q2, q3 = n // 4, n // 2, 3 * n // 4
    ecg_new = np.stack([
        np.concatenate([ecg[0:q1,0], ecg[q1:q2,3], ecg[q2:q3,6],  ecg[q3:n,9]]),
        np.concatenate([ecg[0:q1,1], ecg[q1:q2,4], ecg[q2:q3,7],  ecg[q3:n,10]]),
        np.concatenate([ecg[0:q1,2], ecg[q1:q2,5], ecg[q2:q3,8],  ecg[q3:n,11]]),
        ecg[:, 1],
    ], axis=1).T   # (4, 1000)

    passed = sum(1 for v in checks.values() if v[0])
    total  = len(checks)
    title  = f"{name}  |  Predicted: {pred} ({conf:.2f})"
    if true_label: title += f"  |  True: {true_label}"
    title += f"  |  Clinical: {passed}/{total}"

    ecg_plot.plot(ecg_new, sample_rate=100, title=title,
                  columns=1, lead_index=["I", "II", "III", "II Ref"])
    ax = plt.gca()
    bottoms, tops, yfracs, h_row = _row_geom(ax)

    # Layer 1 — light SHAP time bands (temporal context across all rows)
    for rank, seg in enumerate(top3_segs):
        ax.axvspan(seg * 0.4, (seg + 1) * 0.4,
                   color=_SC3[rank], alpha=0.08, zorder=4)

    # Layer 2 — colored Rectangle borders on lead cells + double-mark fill
    for lead_name, (col, row) in LEAD_CELL_MAP.items():
        is_m = lead_name in med_leads
        is_h = lead_name in top3_hyb
        if not (is_m or is_h):
            continue
        ec, ls, lw, fa = _role(lead_name, med_leads, top3_hyb)
        x0, x1 = COL_X[col]
        ax.add_patch(Rectangle(
            (x0, bottoms[row]), x1 - x0, tops[row] - bottoms[row],
            linewidth=lw, edgecolor=ec, facecolor="none",
            linestyle=ls, zorder=7, clip_on=True,
        ))
        # Double-mark: shade cell when SHAP segment falls in this column
        if is_h:
            for rank, seg in enumerate(top3_segs):
                if min(int(seg * 0.4 / 2.5), 3) == col:
                    ax.axvspan(seg * 0.4, (seg + 1) * 0.4,
                               ymin=yfracs[row][0], ymax=yfracs[row][1],
                               color=_SC3[rank], alpha=fa, zorder=5)

    # Rhythm strip (Lead II, row 3, spans full 10 s)
    is_II_m = "II" in med_leads
    is_II_h = "II" in top3_hyb
    if is_II_m or is_II_h:
        ec, ls, lw, fa = _role("II", med_leads, top3_hyb)
        ax.add_patch(Rectangle(
            (0, bottoms[3]), 10, tops[3] - bottoms[3],
            linewidth=lw, edgecolor=ec, facecolor="none",
            linestyle=ls, zorder=7, clip_on=True,
        ))
        if is_II_h:
            for rank, seg in enumerate(top3_segs):
                ax.axvspan(seg * 0.4, (seg + 1) * 0.4,
                           ymin=yfracs[3][0], ymax=yfracs[3][1],
                           color=_SC3[rank], alpha=fa, zorder=5)

    # Lead column labels (same y-offsets as Cell 13)
    for xpos, labels in [
        (2.55, [(-0.6, "aVR"), (-4.1, "aVL"), (-7.6, "aVF")]),
        (5.05, [(-0.6, "V1"),  (-4.1, "V2"),  (-7.6, "V3")]),
        (7.55, [(-0.6, "V4"),  (-4.1, "V5"),  (-7.6, "V6")]),
    ]:
        for ypos, label in labels:
            ax.text(xpos, ypos, label, fontsize=8, color="black")

    ax.text(0.1, bottoms[3] + 0.2, "100 Hz   25.0 mm/s   10.0mm/mV", fontsize=9)

    # R-peaks on rhythm strip
    y_rp = bottoms[3] + h_row * 0.15
    for rp_s in r_peaks_sec:
        ax.plot(rp_s, y_rp, "v", color="steelblue", markersize=5, alpha=0.85, zorder=8)

    # Clinical annotation bar
    parts = []
    if f.get("heart_rate_bpm"):  parts.append(f"HR: {f['heart_rate_bpm']:.0f} bpm")
    if f.get("qrs_duration_ms"): parts.append(f"QRS: {f['qrs_duration_ms']:.0f} ms")
    if f.get("pr_interval_ms"):  parts.append(f"PR: {f['pr_interval_ms']:.0f} ms")
    if f.get("rr_std_ms"):       parts.append(f"RR SD: {f['rr_std_ms']:.1f} ms")
    if f.get("rr_rmssd_ms"):     parts.append(f"RMSSD: {f['rr_rmssd_ms']:.1f} ms")
    ax.text(0.5, -0.07, "  |  ".join(parts), transform=ax.transAxes,
            fontsize=8, ha="center",
            bbox=dict(facecolor="lightyellow", alpha=0.85, pad=3, edgecolor="none"))

    # Legend
    ax.legend(handles=[
        mpatches.Patch(facecolor=_SC3[0], alpha=0.4, label="SHAP top-1 segment"),
        mpatches.Patch(facecolor=_SC3[1], alpha=0.3, label="SHAP top-2 segment"),
        mpatches.Patch(facecolor=_SC3[2], alpha=0.3, label="SHAP top-3 segment"),
        mpatches.Patch(facecolor="none", edgecolor="#d73027", linewidth=2,
                       label="Medical expected + model used"),
        mpatches.Patch(facecolor="none", edgecolor="#f46d43", linewidth=1.5,
                       label="Model-identified (unexpected)"),
        mpatches.Patch(facecolor="none", edgecolor="#888888", linewidth=1, linestyle="--",
                       label="Medical expected (not top-3)"),
        Line2D([0], [0], marker="v", color="w", markerfacecolor="steelblue",
               markersize=7, label="R-peak (II Ref)"),
    ], loc="upper right", fontsize=7, framealpha=0.9)

    if save:
        path = os.path.join(HYBRID_DIR, f"hybrid12lead_{name}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {path}")
    ecg_plot.show()
    plt.close()


# ── Plot B: Enhanced diagnostic detail ───────────────────────────────────────

def plot_hybrid_detail(name, info, h_res, save=True):
    pred        = info["pred_class"]
    conf        = info["pred_confidence"]
    true_label  = info["true_label"]
    f           = info["clinical_features"]
    checks      = info["clinical_checks"]
    r_peaks_sec = np.array(f.get("r_peaks", []), dtype=int) / 500.0
    shap_vals   = info["shap_values"]           # (25,) full temporal SHAP

    scores      = h_res["scores"]               # (12,) lead contributions
    top3_segs   = h_res["top3_segs"]
    top3_hyb_l  = h_res["top3_names"]           # ordered list
    top3_hyb    = set(top3_hyb_l)
    med_leads   = DIAG_LEADS_H.get(pred, {"II"})

    # Show union of medical + hybrid leads, in canonical lead order
    show_leads = [l for l in LEAD_NAMES if l in (med_leads | top3_hyb)]
    lead_idxs  = [LEAD_NAMES.index(l) for l in show_leads]
    n_show     = len(show_leads)

    sig500 = info["signal"]                      # (12, 5000) preprocessed
    t      = np.linspace(0, 10, 5000)

    fig, axes = plt.subplots(
        n_show + 2, 1,
        figsize=(14, 2.5 * n_show + 4.5),
        gridspec_kw={"height_ratios": [2.5] * n_show + [1.8, 1.5]},
    )

    passed = sum(1 for v in checks.values() if v[0])
    total  = len(checks)
    title  = f"{name} — Hybrid SHAP  |  Predicted: {pred} ({conf:.2f})"
    if true_label: title += f"  |  True: {true_label}"
    title += f"  |  Clinical: {passed}/{total}"
    fig.suptitle(title, fontsize=11, fontweight="bold", y=1.01)

    for ax_i, (lead_name, lead_idx) in enumerate(zip(show_leads, lead_idxs)):
        ax  = axes[ax_i]
        sig = sig500[lead_idx]
        is_m = lead_name in med_leads
        is_h = lead_name in top3_hyb

        if is_m and is_h:
            lc  = "#d73027"
            tag = " [med+model]"
        elif is_h:
            rank_h = top3_hyb_l.index(lead_name) + 1
            lc  = "#f46d43"
            tag = f" [model #{rank_h}]"
        else:
            lc  = "#888888"
            tag = " [med only]"

        ax.plot(t, sig, color=(0, 0, 0.7), linewidth=0.8)
        ax.set_ylabel(f"{lead_name}{tag}", fontsize=8, color=lc, fontweight="bold")
        ax.set_xlim(0, 10)
        ax.tick_params(labelsize=7)

        # SHAP segment bands
        for rank, seg in enumerate(top3_segs):
            alpha = 0.48 if (is_m or is_h) else 0.14
            ax.axvspan(seg * 0.4, (seg + 1) * 0.4,
                       color=_SC3[rank], alpha=alpha, zorder=3)

        # R-peak markers
        for rp_s in r_peaks_sec:
            if 0 <= rp_s <= 10:
                ax.axvline(rp_s, color="steelblue", alpha=0.30, linewidth=0.7, zorder=2)

        # Class-specific clinical annotations
        if pred == "1dAVb" and lead_name == "II":
            pr_ms = f.get("pr_interval_ms")
            if pr_ms:
                for rp_s in r_peaks_sec[:3]:
                    p0 = max(0.0, rp_s - pr_ms / 1000.0 - 0.04)
                    p1 = max(0.0, rp_s - 0.04)
                    ya = sig.max() * 0.85
                    ax.annotate("", xy=(p1, ya), xytext=(p0, ya),
                                arrowprops=dict(arrowstyle="<->", color="#2c7bb6", lw=1.2))
                    ax.text((p0 + p1) / 2, ya + 0.05,
                            f"PR={pr_ms:.0f}ms", ha="center", fontsize=7, color="#2c7bb6")

        elif pred in ("LBBB", "RBBB"):
            qrs_ms = f.get("qrs_duration_ms")
            if qrs_ms:
                for rp_s in r_peaks_sec[:2]:
                    q0 = rp_s - 0.02
                    q1 = rp_s + qrs_ms / 1000.0
                    ya = sig.max() * 0.90
                    ax.annotate("", xy=(q1, ya), xytext=(q0, ya),
                                arrowprops=dict(arrowstyle="<->", color="#1a9641", lw=1.2))
                    ax.text((q0 + q1) / 2, ya + 0.05,
                            f"QRS={qrs_ms:.0f}ms", ha="center", fontsize=7, color="#1a9641")

        elif pred == "AFIB" and lead_name == "II":
            rmssd = f.get("rr_rmssd_ms")
            if rmssd:
                ax.set_title(f"  RMSSD={rmssd:.1f} ms (irregular RR -> AFIB)",
                             fontsize=7, loc="left", color="#7b2d8b")

        elif pred == "AFLT" and lead_name in ("II", "III", "aVF"):
            hr = f.get("heart_rate_bpm")
            if hr:
                ax.set_title(f"  HR={hr:.0f} bpm - look for sawtooth flutter waves",
                             fontsize=7, loc="left", color="#7b2d8b")

    # Bottom panel 1 — temporal SHAP bar chart
    ax_shap = axes[-2]
    t_ctr = np.arange(N_SEGMENTS) * 0.4 + 0.2
    ax_shap.bar(t_ctr, shap_vals, width=0.35, color="#cccccc", alpha=0.7)
    for rank, seg in enumerate(top3_segs):
        ax_shap.bar(seg * 0.4 + 0.2, shap_vals[seg], width=0.35,
                    color=_SC3[rank], alpha=0.85,
                    label=f"Top-{rank+1}: {seg*0.4:.1f}s")
    ax_shap.set_xlim(0, 10)
    ax_shap.set_xlabel("Time (s)", fontsize=8)
    ax_shap.set_ylabel("Temporal SHAP", fontsize=8)
    ax_shap.legend(fontsize=7, loc="upper left")
    ax_shap.tick_params(labelsize=7)
    ax_shap.axhline(0, color="black", linewidth=0.5)

    # Bottom panel 2 — hybrid lead importance bar
    ax_lead = axes[-1]
    sc_max = max(np.abs(scores).max(), 1e-8)
    x_pos  = np.arange(12)
    bar_c  = []
    for ln in LEAD_NAMES:
        if ln in med_leads and ln in top3_hyb: bar_c.append("#d73027")
        elif ln in top3_hyb:                   bar_c.append("#f46d43")
        elif ln in med_leads:                  bar_c.append("#aaaaaa")
        else:                                  bar_c.append("#dddddd")
    ax_lead.bar(x_pos, scores / sc_max, color=bar_c, alpha=0.75, width=0.7)
    ax_lead.axhline(0, color="black", linewidth=0.5)
    ax_lead.set_xticks(x_pos)
    ax_lead.set_xticklabels(LEAD_NAMES, fontsize=8)
    ax_lead.set_ylabel("Hybrid lead score\n(normalised)", fontsize=7)
    ax_lead.tick_params(labelsize=7)
    ax_lead.legend(handles=[
        mpatches.Patch(color="#d73027", label="Medical + model"),
        mpatches.Patch(color="#f46d43", label="Model-only"),
        mpatches.Patch(color="#aaaaaa", label="Medical-only"),
    ], fontsize=7, loc="upper right")

    plt.tight_layout()
    if save:
        path = os.path.join(HYBRID_DIR, f"hybrid_detail_{name}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {path}")
    plt.show()
    plt.close()


# ── Generate all figures ──────────────────────────────────────────────────────

print("Generating hybrid 12-lead figures...")
for name, info in samples.items():
    plot_hybrid_12lead(name, info, hybrid_results[name])

print("\nGenerating hybrid detail figures...")
for name, info in samples.items():
    plot_hybrid_detail(name, info, hybrid_results[name])

print(f"\nAll hybrid figures saved to: {HYBRID_DIR}")


In [ ]:
try:
    import ecg_plot
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "ecg-plot", "-q"], check=True)
    import ecg_plot

def load_raw_signal(name, sample_type):
    base = "../../../data/validation" if sample_type == "validation" else "../../../data/test"
    return np.load(f"{base}/{name}/{name}.npy").astype(np.float32)


def plot_12lead_shap(name, info, save=True):
    pred        = info["pred_class"]
    conf        = info["pred_confidence"]
    true_label  = info["true_label"]
    f           = info["clinical_features"]
    checks      = info["clinical_checks"]
    r_peaks_500 = np.array(f.get("r_peaks", []), dtype=int)
    r_peaks_sec = r_peaks_500 / 500.0
    raw_sig = load_raw_signal(name, info["type"])
    ecg     = raw_sig.T   # (1000, 12)
    top2        = info["shap_top_segments"][:2]
    shap_colors = ["#d73027", "#f46d43"]
    shap_alphas = [0.50, 0.35]

    n = ecg.shape[0]
    q1, q2, q3 = n//4, n//2, 3*n//4
    lead_I_new   = np.concatenate([ecg[0:q1,0],  ecg[q1:q2,3],  ecg[q2:q3,6],  ecg[q3:n,9]])
    lead_II_new  = np.concatenate([ecg[0:q1,1],  ecg[q1:q2,4],  ecg[q2:q3,7],  ecg[q3:n,10]])
    lead_III_new = np.concatenate([ecg[0:q1,2],  ecg[q1:q2,5],  ecg[q2:q3,8],  ecg[q3:n,11]])
    lead_IV_new  = ecg[:, 1]
    ecg_new = np.stack([lead_I_new, lead_II_new, lead_III_new, lead_IV_new], axis=1).T

    passed = sum(1 for v in checks.values() if v[0])
    total  = len(checks)
    title  = f"{name}  |  Predicted: {pred} ({conf:.2f})"
    if true_label: title += f"  |  True: {true_label}"
    title += f"  |  Clinical checks: {passed}/{total}"

    ecg_plot.plot(ecg_new, sample_rate=100, title=title,
                  columns=1, lead_index=['I', 'II', 'III', 'II Ref'])
    ax = plt.gca()

    ax.text(2.55, -0.6,  'aVR', fontsize=8, color='black')
    ax.text(2.55, -4.1,  'aVL', fontsize=8, color='black')
    ax.text(2.55, -7.6,  'aVF', fontsize=8, color='black')
    ax.text(5.05, -0.6,  'V1',  fontsize=8, color='black')
    ax.text(5.05, -4.1,  'V2',  fontsize=8, color='black')
    ax.text(5.05, -7.6,  'V3',  fontsize=8, color='black')
    ax.text(7.55, -0.6,  'V4',  fontsize=8, color='black')
    ax.text(7.55, -4.1,  'V5',  fontsize=8, color='black')
    ax.text(7.55, -7.6,  'V6',  fontsize=8, color='black')
    ax.text(0.1, ax.get_ylim()[0] + 0.2, '100 Hz   25.0 mm/s   10.0mm/mV', fontsize=9)

    for rank, seg in enumerate(top2):
        ax.axvspan(seg * 0.4, (seg + 1) * 0.4,
                   color=shap_colors[rank], alpha=shap_alphas[rank], zorder=5)

    y_rhythm = ax.get_ylim()[0] + abs(ax.get_ylim()[0]) * 0.15
    for rp_s in r_peaks_sec:
        ax.plot(rp_s, y_rhythm, "v", color="steelblue", markersize=5, alpha=0.8, zorder=6)

    parts = []
    if f.get("heart_rate_bpm"):  parts.append(f"HR: {f['heart_rate_bpm']:.0f} bpm")
    if f.get("qrs_duration_ms"): parts.append(f"QRS: {f['qrs_duration_ms']:.0f} ms")
    if f.get("pr_interval_ms"):  parts.append(f"PR: {f['pr_interval_ms']:.0f} ms")
    if f.get("rr_std_ms"):       parts.append(f"RR SD: {f['rr_std_ms']:.1f} ms")
    if f.get("rr_rmssd_ms"):     parts.append(f"RMSSD: {f['rr_rmssd_ms']:.1f} ms")
    ax.text(0.5, -0.07, "  |  ".join(parts), transform=ax.transAxes,
            fontsize=8, ha="center",
            bbox=dict(facecolor="lightyellow", alpha=0.85, pad=3, edgecolor="none"))

    legend_els = [
        Patch(facecolor="#d73027", alpha=0.6, label="SHAP top-1 region"),
        Patch(facecolor="#f46d43", alpha=0.5, label="SHAP top-2 region"),
        Line2D([0],[0], marker="v", color="w", markerfacecolor="steelblue",
               markersize=7, label="R-peak (rhythm strip)"),
    ]
    ax.legend(handles=legend_els, loc="upper right", fontsize=8, framealpha=0.9)

    if save:
        path = os.path.join(UNIFIED_DIR, f"raw12lead_{name}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        print(f"Saved: {path}")
    ecg_plot.show()
    plt.close()


print("Generating standard 12-lead + SHAP visualizations...\n")
for name, info in samples.items():
    plot_12lead_shap(name, info)
print(f"\nAll 12-lead figures saved to: {UNIFIED_DIR}")
